# 第 2 章 · LCEL 与链式编排：LangChain 的管道美学

> 本章目标：
> 1. 掌握 **LCEL（LangChain Expression Language）**——用 `|` 管道符把组件连成声明式工作流；
> 2. 学会提示词模板、输出解析、并行分支与结构化输出；
> 3. 用 LCEL 搭建一个迷你 RAG 管道，理解"检索增强生成"的骨架。

---

## 1. 为什么需要 LCEL？

把 LLM 应用拆开看，几乎都是同一个模式：**输入 → 组装提示词 → 调模型 → 解析输出 → （可能再来一轮）**。LCEL 把每个环节抽象为 Runnable，用 Unix 管道一样的语法串联：

```python
chain = prompt | llm | parser
result = chain.invoke({"topic": "量子计算"})
```

```mermaid
flowchart LR
    I["输入 dict<br/>{topic: ...}"] --> P["ChatPromptTemplate<br/>渲染提示词"]
    P --> L["ChatModel<br/>调用 LLM"]
    L --> O["StrOutputParser<br/>提取纯文本"]
    O --> R["最终结果 str"]
    style P fill:#e6f4ea,stroke:#34a853
    style L fill:#e8f0fe,stroke:#4285f4
    style O fill:#fef7e0,stroke:#fbbc04
```

| 你得到的好处 | 说明 |
|---|---|
| **统一接口** | 拼出来的 chain 本身也是 Runnable：`invoke/stream/batch` 全都自动支持 |
| **流式穿透** | 链条末端才开始消费，但流式从 LLM 第一个 token 就开始了 |
| **声明式** | 管道即文档——看一眼就知道数据怎么流 |
| **可组合** | chain 可以当零件塞进更大的 chain |

先准备模型（沿用第 1 章的 DeepSeek 配置）：


In [1]:
import os
assert os.environ.get("DEEPSEEK_API_KEY"), "请先设置环境变量 DEEPSEEK_API_KEY"

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com",
    temperature=0,
)
print("✅ 模型就绪")


✅ 模型就绪


---

## 2. 提示词模板：ChatPromptTemplate

`ChatPromptTemplate` 把"消息列表"参数化——`{变量}` 在 invoke 时被替换：


In [2]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是{role}。回答风格：{style}。"),
    ("human", "{question}"),
])

# 单独看渲染效果（不调模型）
rendered = prompt.invoke({"role": "美食评论家", "style": "毒舌但专业", "question": "评价一下螺蛳粉"})
for m in rendered.messages:
    print(f"[{m.type}] {m.content}")


[system] 你是美食评论家。回答风格：毒舌但专业。
[human] 评价一下螺蛳粉


---

## 3. 第一条链：prompt | llm | parser

`StrOutputParser` 把 `AIMessage` "剥壳"成纯字符串。三个组件用 `|` 连接即成链：


In [3]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

answer = chain.invoke({
    "role": "美食评论家",
    "style": "毒舌但专业，三句话以内",
    "question": "评价一下螺蛳粉",
})
print(answer)
print("\n类型：", type(answer).__name__)   # str，而不是 AIMessage


这碗螺蛳粉，酸笋的“臭”像被卡车碾过的发酵池，汤底咸到能腌咸菜，粉条软得像失恋者的膝盖——但偏偏辣油香得让人想犯罪，真是又脏又上头的街头美学。

类型： TextAccessor


> 💡 整条链也是 Runnable——试试 `chain.stream(...)`，你会看到流式输出**自动穿透整条链**。这就是统一接口的复利。

---

## 4. 数据流三剑客：Passthrough / Lambda / Parallel

真实管道很少是直线，LCEL 提供三个"管件"：

| 管件 | 作用 | 类比 |
|---|---|---|
| `RunnablePassthrough()` | 原样传递输入 | 直通水管 |
| `RunnableLambda(func)` | 把任意函数变成 Runnable | 自定义加工站 |
| `RunnableParallel({...})` | 同一输入分发给多个分支，结果合并为 dict | 分水器 |

```mermaid
flowchart TD
    I["输入: {question}"] --> PP{"RunnableParallel"}
    PP --> A["分支1: 直接回答"]
    PP --> B["分支2: 列出要点"]
    PP --> C["分支3: 给出类比"]
    A --> M["合并成 dict"]
    B --> M
    C --> M
    M --> F["RunnableLambda<br/>格式化成最终答案"]
    style PP fill:#e6f4ea,stroke:#34a853,stroke-width:2px
```


In [4]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough

qa_prompt = ChatPromptTemplate.from_template("用一句话回答：{question}")
points_prompt = ChatPromptTemplate.from_template("针对「{question}」，列出 2 个关键要点，每点不超过 15 字。")
analogy_prompt = ChatPromptTemplate.from_template("为「{question}」的主题想一个生活化的类比，一句话。")

parallel = RunnableParallel({
    "直接回答": qa_prompt | llm | StrOutputParser(),
    "关键要点": points_prompt | llm | StrOutputParser(),
    "生活类比": analogy_prompt | llm | StrOutputParser(),
})

def format_report(d: dict) -> str:
    lines = [f"【{k}】{v}" for k, v in d.items()]
    return "\n\n".join(lines)

full_chain = parallel | RunnableLambda(format_report)

print(full_chain.invoke({"question": "为什么程序员要学点心理学？"}))


【直接回答】因为编程本质上是与人协作和沟通的系统工程，懂心理学才能更好地理解需求、设计体验，并保护自己的心智健康。

【关键要点】1. 理解需求，减少返工  
2. 沟通顺畅，协作高效

【生活类比】**“改代码就像谈恋爱——你以为是逻辑问题，其实全是情绪问题。”**


> 🔍 三个分支**并发执行**（batch 语义），总耗时 ≈ 最慢的分支，而非三者之和。对比 ADK 第 4 章的 `ParallelAgent`——思想相同，但 LCEL 操作的是"数据"，ADK 操作的是"Agent"。

---

## 5. 结构化输出：with_structured_output

业务系统需要的往往不是文本，而是**能进数据库的对象**。`with_structured_output` 让模型按 Pydantic schema 输出。

> ⚠️ **DeepSeek 兼容性细节**：`with_structured_output` 的 `method` 参数决定底层通道——`"json_schema"`（新版默认，DeepSeek 不支持）、`"json_mode"`（DeepSeek 支持但不带 schema 约束）、`"function_calling"`（走工具调用通道，**DeepSeek 支持**）。因此本教程显式指定 `method="function_calling"`——这也是接非 OpenAI 模型时值得记住的排错经验。


In [5]:
from pydantic import BaseModel, Field

class Recipe(BaseModel):
    name: str = Field(description="菜名")
    difficulty: int = Field(description="难度，1-5")
    time_minutes: int = Field(description="预计耗时（分钟）")
    steps: list[str] = Field(description="关键步骤，3-5 步")

recipe_llm = llm.with_structured_output(Recipe, method="function_calling")

recipe = recipe_llm.invoke("教我做一个适合新手的番茄炒蛋")
print(type(recipe).__name__)          # Recipe —— 直接是 Pydantic 对象！
print(f"菜名：{recipe.name} | 难度：{'⭐' * recipe.difficulty} | 耗时：{recipe.time_minutes} 分钟")
for i, step in enumerate(recipe.steps, 1):
    print(f"  {i}. {step}")


Recipe
菜名：番茄炒蛋 | 难度：⭐ | 耗时：15 分钟
  1. 将番茄洗净切块，鸡蛋打入碗中加少许盐打散备用
  2. 热锅倒油，油热后倒入蛋液，快速翻炒至凝固后盛出
  3. 锅中留底油，放入番茄块翻炒至出汁，加少许糖和盐调味
  4. 将炒好的鸡蛋倒回锅中，与番茄翻炒均匀即可出锅


> 💡 与 ADK 的 `output_schema` 对照：ADK 把结构化输出绑定在 Agent 上，LangChain 绑定在**模型实例**上（`recipe_llm` 是一个"只会输出 Recipe 的新模型"）。后者更适合"同一 Agent 多种输出形态"的场景。

---

## 6. 综合实战：迷你 RAG 管道

**RAG（检索增强生成）** 是 LLM 应用的头号模式：先从知识库**检索**相关内容，再把内容**塞进提示词**让模型基于它回答——解决"模型不知道你的私有数据"的问题。

```mermaid
flowchart LR
    Q["用户问题"] --> RET["🔍 Retriever<br/>检索相关文档"]
    RET --> CTX["📄 相关文档片段"]
    CTX --> PR["PromptTemplate<br/>上下文+问题"]
    Q --> PR
    PR --> LLM["LLM"]
    LLM --> ANS["有据可查的回答"]
    style RET fill:#e6f4ea,stroke:#34a853,stroke-width:2px
```

> 📌 为了不引入向量数据库依赖，下面用 `RunnableLambda` **模拟**检索器（演示环境按关键词匹配）。生产中的标准替换件：`FAISS` / `Chroma` / `pgvector` + Embedding 模型，接口完全一样。


In [6]:
# ---------- 模拟的知识库 ----------
KNOWLEDGE_BASE = [
    {"doc": "公司年假政策：入职满 1 年享有 10 天年假，满 3 年 15 天，满 5 年 20 天。年假可顺延至次年 3 月底。", "tag": "年假"},
    {"doc": "报销制度：单笔 500 元以下由直属主管审批，500-5000 元需部门总监审批，5000 元以上需 VP 审批。发票需在消费后 30 天内提交。", "tag": "报销"},
    {"doc": "远程办公政策：每周可申请最多 2 天远程办公，需提前一天在 OA 系统报备，核心会议日（周二）必须到岗。", "tag": "远程办公"},
]

def mock_retriever(question: str) -> str:
    """模拟检索器：按关键词命中知识库（生产中替换为向量检索）。"""
    hits = [item["doc"] for item in KNOWLEDGE_BASE if item["tag"] in question]
    return "\n".join(hits) if hits else "（知识库中没有相关内容）"

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是公司制度问答助手。严格基于【参考资料】回答；资料中没有的信息要明确说不知道，禁止编造。\n\n【参考资料】\n{context}"),
    ("human", "{question}"),
])

rag_chain = (
    {"context": RunnableLambda(mock_retriever), "question": RunnablePassthrough()}  # dict 语法 = 隐式 Parallel
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("Q1:", rag_chain.invoke("我入职 4 年了，有多少天年假？"))
print("─" * 50)
print("Q2:", rag_chain.invoke("公司的健身房几点关门？"))   # 知识库没有 → 应该诚实说不知道


Q1: 根据公司年假政策，入职满 3 年享有 15 天年假。您入职 4 年，符合“满 3 年”的条件，因此您的年假为 **15 天**。
──────────────────────────────────────────────────


Q2: 关于公司健身房的开放时间，我目前的知识库中没有相关信息，无法提供准确答案。建议您查看公司内部公告、员工手册，或咨询行政/前台部门获取具体安排。


> 🔍 两个细节值得品味：
> 1. `{"context": ..., "question": ...}` 这个 dict 是 LCEL 的语法糖——字典中的每个 Runnable 并行执行，凑成新 dict 传给下一步；
> 2. Q2 的回答质量取决于**提示词里的"禁止编造"**——RAG 系统的工程细节一半在检索，一半在提示词纪律。

---

## 7. LCEL 的边界：什么时候该换 LangGraph？

LCEL 擅长**有向无环**的数据流（DAG），但一旦你需要：

| 需求 | LCEL | LangGraph |
|---|---|---|
| 循环（反复修正直到满意） | ❌ 做不到 | ✅ 回边天然支持 |
| 复杂条件分支+状态累积 | 💛 别扭 | ✅ State + 条件边 |
| 多轮对话记忆 | ❌ 手动管理 | ✅ Checkpointer |
| 人机协同（暂停等人审批） | ❌ | ✅ interrupt |
| 多 Agent 协作 | ❌ | ✅ |

> 📌 **记忆口诀**：LCEL 是**流水线**，LangGraph 是**电路板**（可以接回路线）。ADK 世界里没有 LCEL 的对应物——ADK 用"工作流 Agent"同时覆盖了流水线（Sequential）和回路（Loop）两种需求。

---

## 8. 与 ADK 对照 🔄

| LCEL 概念 | ADK 对应物 | 差异 |
|---|---|---|
| `prompt \| llm \| parser` | 没有直接对应（ADK 不做数据流管道） | LCEL 是 LangChain 独有美学 |
| `RunnableParallel` | `ParallelAgent` | 数据并行 vs Agent 并行 |
| `ChatPromptTemplate {var}` | `instruction {state}` 插值 | 模板渲染时机不同 |
| `with_structured_output` | `output_schema` | 绑定模型 vs 绑定 Agent |
| 模拟检索器 / 向量库集成 | Vertex AI RAG / 自定义 Tool | LangChain 检索生态更丰富 |

---

## 📌 本章要点回顾

- LCEL = 用 `|` 把 Runnable 连成**声明式数据流管道**，链本身也是 Runnable；
- 三剑客：`Passthrough`（直通）、`Lambda`（自定义函数）、`Parallel`（扇出合并）；
- `with_structured_output(Pydantic)` 一行获得类型安全的结构化输出；
- RAG 骨架 = 检索器 + 提示词纪律 + 生成，LCEL 表达它只要几行；
- 需要**循环/状态/记忆/人机协同**时，就是 LangGraph 的战场了。

> ➡️ 下一章：[03-Agent与工具调用](03-Agent与工具调用.ipynb) —— `create_agent` 与 LangChain 1.x 的 Agent 范式。


---

## 🧪 本章练习

### 1. 从提示词到可复用链（基础）

构建“代码评审摘要”链：`ChatPromptTemplate` 接收语言、代码与关注点，经模型后由解析器输出文本。要求模板变量缺失时尽早报错，同一条链可 `invoke` 也可 `stream`，并用三种不同代码片段验证输出结构稳定。

### 2. 并行分析与数据整形（进阶）

组合 `RunnablePassthrough`、`RunnableLambda` 和 `RunnableParallel`，对一段客户反馈并行提取情绪、主题和紧急度，再把原文与三个结果汇总成结构化报告。画出每一步的输入/输出形状，确保任一分支失败时能给出可解释的降级结果。

### 3. 强类型结构化输出（进阶）

用 Pydantic 定义工单对象，包含分类、优先级、摘要、建议负责人和置信度；通过 `with_structured_output` 获得实例并执行额外业务校验。对模糊输入和缺少关键信息的输入，明确是拒绝、补问还是使用默认值，并为选择写测试。

### 4. 把迷你 RAG 做到可验收（工程）

扩展本章 RAG：文档分块需带来源 ID，回答必须附引用；检索不到证据时明确说不知道，不能编造。准备至少十个问题（含不可回答问题），统计引用正确率和拒答正确率。可在脚本、API 或小型 Web 界面中完成，不限定 Notebook。

### 5. 识别 LCEL 的边界（跨章节）

为练习 4 增加“答案低分则改写查询并重新检索，最多三次”的需求。先说明为何纯 LCEL 表达会变得困难，再画出 LangGraph 状态图；同时给出 ADK `LoopAgent` 版本的构件映射，比较三种表达方式的可读性与可控性。
